# 04 — Bonus: Görev Atama (Task Assignment)

> Bu notebook workshop'ta anlatılmaz. Ev çalışması olarak incelemeniz için hazırlanmıştır.

**Öğrenme hedefleri:**
1. Binary ve sürekli değişkenleri aynı modelde kullanmak (MILP)
2. Min-max objective'i doğrusallaştırmayı öğrenmek
3. Gantt grafiği ile sonucu görselleştirmek

**Tahmini süre:** ~30-45 dakika

In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip install -q pulp
    !git clone -q https://github.com/Meliksahcaliskan/ILP_Workshop.git
    sys.path.insert(0, "ILP_Workshop")

In [ ]:
import pulp
import pandas as pd
import matplotlib.pyplot as plt
from utils.plotting import plot_gantt

---
## Problem Tanımı

8 görev 3 işlemciye atanacak.  
Her görevin her işlemcideki **çalışma süresi** farklı.  
**Amaç:** Tüm görevlerin biteceği en geç zamanı (**makespan**) minimize et.

Bu tip probleme **Scheduling / Makespan Minimization** denir.  
İşlemci yük dengeleme, paralel derleme, bulut iş yükü dağıtımı bunun örnekleri.

In [ ]:
# Görev x İşlemci süresi tablosu (deterministik)
# proc_times[job][machine] = süre
proc_times = {
    0: {0: 3, 1: 5, 2: 2},
    1: {0: 7, 1: 4, 2: 6},
    2: {0: 2, 1: 3, 2: 8},
    3: {0: 5, 1: 6, 2: 3},
    4: {0: 4, 1: 2, 2: 5},
    5: {0: 6, 1: 7, 2: 4},
    6: {0: 3, 1: 4, 2: 6},
    7: {0: 8, 1: 3, 2: 2},
}

jobs     = list(proc_times.keys())
machines = list(proc_times[0].keys())

# Tablo olarak göster
df = pd.DataFrame(proc_times).T
df.index.name = "Görev"
df.columns = [f"İşlemci {m}" for m in machines]
print("Çalışma Süreleri Tablosu:")
print(df)

---
## Neden `min max` Doğrudan Yazılamaz?

ILP (LP) modelinde objective fonksiyonu **doğrusal** olmak zorunda.  
`min max(...)` ifadesi doğrusal değil — `max` bir fonksiyon değil, bir operatör.

**Çözüm: Yardımcı değişken `C_max`**

> `C_max` = tüm işlemcilerin biteceği en geç zaman

Bunu modele şöyle dahil ederiz:
- `minimize C_max`
- `Σ_j p[j,m] · x[j,m] ≤ C_max`  (her işlemci için)

`C_max` aşağıdan sıkıştırıldığı için minimize ederken solver onu olabildiğince küçük tutacak — bu da tam olarak makespan'i minimize etmek demek!

---
## Model

**Değişkenler:**
- `x[j, m] ∈ {0, 1}` — görev `j`, işlemci `m`'ye atandıysa 1
- `C_max ≥ 0` (sürekli) — makespan

**Kısıtlar:**
1. Her görev tam olarak bir işlemciye: `Σ_m x[j,m] = 1`
2. Her işlemcinin yükü makespan'i aşmaz: `Σ_j p[j,m] · x[j,m] ≤ C_max`

In [ ]:
prob = pulp.LpProblem("task_assignment", pulp.LpMinimize)

x = {(j, m): pulp.LpVariable(f"x_{j}_{m}", cat="Binary")
     for j in jobs for m in machines}

C_max = pulp.LpVariable("C_max", lowBound=0, cat="Continuous")

# Objective
prob += C_max

# Her görev tam olarak bir işlemciye
for j in jobs:
    prob += pulp.lpSum(x[j, m] for m in machines) == 1

# Her işlemcinin yükü makespan'i aşamaz
for m in machines:
    prob += pulp.lpSum(proc_times[j][m] * x[j, m] for j in jobs) <= C_max

prob.solve(pulp.PULP_CBC_CMD(msg=0))

print(f"Durum   : {pulp.LpStatus[prob.status]}")
print(f"Makespan: {pulp.value(C_max):.1f}")

print("\nAtama:")
assignment = {}
for j in jobs:
    for m in machines:
        if pulp.value(x[j, m]) > 0.5:
            assignment[j] = m
            load = proc_times[j][m]
            print(f"  Görev {j} → İşlemci {m}  (süre: {load})")

---
## Gantt Grafiği

In [ ]:
fig = plot_gantt(assignment, proc_times, num_machines=len(machines))
plt.show()

# İşlemci yükleri
print("\nİşlemci yükleri:")
for m in machines:
    load = sum(proc_times[j][m] for j in jobs if assignment.get(j) == m)
    print(f"  İşlemci {m}: {load}")

---
## 🔧 Değiştir-Gözlemle Görevleri

### Görev 1 — İşlemci Sayısını Değiştir

> İşlemci sayısını 4'e çıkar. Makespan nasıl değişir?

In [ ]:
# İşlemci sayısı 4 olacak şekilde proc_times genişlet
proc_times_4m = {
    j: {0: proc_times[j][0], 1: proc_times[j][1], 2: proc_times[j][2],
        3: max(1, proc_times[j][0] + proc_times[j][1] - proc_times[j][2])}
    for j in jobs
}
machines_4 = [0, 1, 2, 3]

prob_g1 = pulp.LpProblem("task_4machines", pulp.LpMinimize)
x_g1 = {(j, m): pulp.LpVariable(f"x_{j}_{m}", cat="Binary")
        for j in jobs for m in machines_4}
cmax_g1 = pulp.LpVariable("C_max", lowBound=0)

prob_g1 += cmax_g1
for j in jobs:
    prob_g1 += pulp.lpSum(x_g1[j, m] for m in machines_4) == 1
for m in machines_4:
    prob_g1 += pulp.lpSum(proc_times_4m[j][m] * x_g1[j, m] for j in jobs) <= cmax_g1

prob_g1.solve(pulp.PULP_CBC_CMD(msg=0))
print(f"3 işlemci makespan: {pulp.value(C_max):.1f}")
print(f"4 işlemci makespan: {pulp.value(cmax_g1):.1f}")

### Görev 2 — Yasak Atama

> Görev 0, İşlemci 2'de çalışamıyor. Bu kısıtı ekle.

In [ ]:
prob_g2 = pulp.LpProblem("task_forbidden", pulp.LpMinimize)
x_g2 = {(j, m): pulp.LpVariable(f"x_{j}_{m}", cat="Binary")
        for j in jobs for m in machines}
cmax_g2 = pulp.LpVariable("C_max", lowBound=0)

prob_g2 += cmax_g2
for j in jobs:
    prob_g2 += pulp.lpSum(x_g2[j, m] for m in machines) == 1
for m in machines:
    prob_g2 += pulp.lpSum(proc_times[j][m] * x_g2[j, m] for j in jobs) <= cmax_g2

# YASAK ATAMA
prob_g2 += x_g2[0, 2] == 0

prob_g2.solve(pulp.PULP_CBC_CMD(msg=0))
print(f"Durum   : {pulp.LpStatus[prob_g2.status]}")
print(f"Makespan: {pulp.value(cmax_g2):.1f}")

### Görev 3 — Objective Değiştir: Toplam Süre

> Makespan yerine toplam işlemci süresini (`Σ_m yük_m`) minimize et. Ne değişir?
> 
> **Tartışma:** Makespan ile toplam süre ne zaman farklı optima verir?

In [ ]:
prob_g3 = pulp.LpProblem("task_total", pulp.LpMinimize)
x_g3 = {(j, m): pulp.LpVariable(f"x_{j}_{m}", cat="Binary")
        for j in jobs for m in machines}

# Toplam süreyi minimize et (NOT: her görev zaten bir işlemciye atanıyor,
# toplam süre deterministik — en kısa sürede çalışan makineye gönder)
prob_g3 += pulp.lpSum(
    proc_times[j][m] * x_g3[j, m] for j in jobs for m in machines
)

for j in jobs:
    prob_g3 += pulp.lpSum(x_g3[j, m] for m in machines) == 1

prob_g3.solve(pulp.PULP_CBC_CMD(msg=0))

total_time = int(pulp.value(prob_g3.objective))
makespan_g3 = max(
    sum(proc_times[j][m] for j in jobs if pulp.value(x_g3[j, m]) > 0.5)
    for m in machines
)

print(f"Toplam süre minimize → toplam süre: {total_time}")
print(f"Ancak bu çözümün makespan'i       : {makespan_g3}")
print(f"Orijinal makespan minimizasyonu    : {int(pulp.value(C_max))}")
print()
print("Yorum: Toplam süreyi minimize etmek her görevi 'en hızlı işlemciye' iter,")
print("bu da o işlemciyi aşırı yükler ve makespan'i artırabilir.")